In [ ]:
#!/usr/bin/env Rscript

## ============================================================
## STEP2: Clustering on precomputed MDS coordinates (A–D)
## - Input : mds_isoMDS_*_YYYYMMDD.rds (from STEP1)
## - Output: cluster labels, NbClust index summary, hclust objects
## - k-range: max = 10, 25, 50 (min = 2)
## ============================================================

suppressPackageStartupMessages({
  library(NbClust)
})

## -----------------------------
## 0. Global settings
## -----------------------------

# ★ STEP1 と同じ ROOT を固定で使用 ★
ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# Type of MDS to use for clustering: "isoMDS" or "cmdscale"
mds_type_for_clustering <- "isoMDS"

# Number of MDS dimensions to use (先頭何次元を使うか）
mds_dim_for_clustering <- 10

# Range of cluster numbers to explore
min_nc <- 2
k_max_list <- c(10, 25, 50)  # 2〜10, 2〜25, 2〜50 の3条件

# Indices to use with NbClust
nb_indices <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")

# Random seed for reproducibility
set.seed(42)

# Date string to embed in output filenames
current_date <- format(Sys.Date(), "%Y%m%d")


## -----------------------------
## 1. Helper functions
## -----------------------------

# Safely extract coordinate matrix from MDS object
get_mds_coords <- function(mds_obj) {
  # isoMDS(MASS) returns a list with $points
  if (is.list(mds_obj) && !is.null(mds_obj$points)) {
    coords <- mds_obj$points
  } else {
    coords <- mds_obj
  }
  coords <- as.matrix(coords)
  return(coords)
}

# ★ STEP1 の命名規則に対応した MDS ファイル探索関数
#   mds_isoMDS_A_YYYYMMDD.rds / mds_cmdscale_A_YYYYMMDD.rds を想定
find_latest_mds_file <- function(input_dir, prefix, mds_type) {
  if (mds_type == "isoMDS") {
    base <- "mds_isoMDS"
  } else if (mds_type == "cmdscale") {
    base <- "mds_cmdscale"
  } else {
    stop("Unknown mds_type_for_clustering: ", mds_type)
  }
  
  pattern <- sprintf("%s_%s_\\d{8}\\.rds", base, prefix)
  files <- list.files(input_dir, pattern = pattern, full.names = TRUE)
  
  if (length(files) == 0L) {
    stop("No MDS file found in ", input_dir,
         " for prefix = ", prefix, ", pattern = ", pattern)
  }
  
  # ファイル名から YYYYMMDD を抜き出して最大を選ぶ
  dates <- sub(sprintf("^%s_%s_(\\d{8})\\.rds$", base, prefix),
               "\\1", basename(files))
  idx <- which.max(dates)
  latest_file <- files[idx]
  
  message("[INFO] Candidate MDS files in ", input_dir, " :")
  print(basename(files))
  message("[INFO] Selected latest MDS file: ", basename(latest_file))
  
  return(latest_file)
}

# Run NbClust for a single index, with error handling
run_nbclust_for_index <- function(data_mat, index_name, min_nc, max_nc) {
  res <- tryCatch(
    {
      NbClust(
        data = data_mat,
        distance = "euclidean",
        min.nc = min_nc,
        max.nc = max_nc,
        method = "ward.D2",
        index = index_name
      )
    },
    error = function(e) {
      message("[WARN] NbClust failed for index = ", index_name,
              " with error: ", conditionMessage(e))
      return(NULL)
    }
  )
  return(res)
}

# Build summary row (best_k, value_best_k, index values for k = min_nc..max_nc)
build_index_summary_row <- function(res, index_name, min_nc, max_nc) {
  k_seq <- min_nc:max_nc
  
  # デフォルトは NA 埋めの 1 行
  row <- data.frame(
    index = index_name,
    best_k = NA_integer_,
    value_best_k = NA_real_,
    t(rep(NA_real_, length(k_seq))),
    check.names = FALSE
  )
  colnames(row) <- c("index", "best_k", "value_best_k",
                     paste0("k", k_seq))
  
  # NbClust 自体が失敗していた場合
  if (is.null(res)) {
    return(row)
  }
  
  ## -----------------------------
  ## 1) Best.nc から best_k / value_best_k を安全に取り出す
  ## -----------------------------
  best_k <- NA_integer_
  value_best_k <- NA_real_
  
  B <- res$Best.nc
  
  if (!is.null(B)) {
    # 行列 or データフレーム
    if (is.matrix(B) || is.data.frame(B)) {
      rn <- rownames(B)
      # 行名に "Number_clusters" / "Value_Index" がある場合
      if (!is.null(rn) && all(c("Number_clusters", "Value_Index") %in% rn)) {
        best_k <- suppressWarnings(as.integer(B["Number_clusters", 1]))
        value_best_k <- suppressWarnings(as.numeric(B["Value_Index", 1]))
      } else {
        # 行名がない or 想定と違う場合 → 1 行目を best_k, 2 行目を value とみなす（保険）
        if (nrow(B) >= 1) {
          best_k <- suppressWarnings(as.integer(B[1, 1]))
        }
        if (nrow(B) >= 2) {
          value_best_k <- suppressWarnings(as.numeric(B[2, 1]))
        }
      }
      
    # ベクトルの場合（今回エラーになっていたケース）
    } else if (is.vector(B)) {
      nm <- names(B)
      if (!is.null(nm) && "Number_clusters" %in% nm) {
        best_k <- suppressWarnings(as.integer(B["Number_clusters"]))
      } else if (length(B) >= 1) {
        best_k <- suppressWarnings(as.integer(B[1]))
      }
      
      if (!is.null(nm) && "Value_Index" %in% nm) {
        value_best_k <- suppressWarnings(as.numeric(B["Value_Index"]))
      } else if (length(B) >= 2) {
        value_best_k <- suppressWarnings(as.numeric(B[2]))
      }
    }
  }
  
  ## -----------------------------
  ## 2) All.index から k ごとの値を取り出す
  ## -----------------------------
  all_idx_values <- res$All.index
  if (!is.null(all_idx_values)) {
    vals <- rep(NA_real_, length(k_seq))
    
    # ベクトルの場合（多くの index がこれ）
    if (is.vector(all_idx_values) && is.null(dim(all_idx_values))) {
      names_k <- suppressWarnings(as.integer(names(all_idx_values)))
      if (any(!is.na(names_k))) {
        # 名前が "2" "3" ... のように付いている場合
        match_idx <- match(names_k, k_seq)
        valid <- which(!is.na(match_idx))
        vals[match_idx[valid]] <- as.numeric(all_idx_values[valid])
      } else {
        # 名前が無い場合 → min_nc:max_nc の順番と仮定
        idx_len <- min(length(all_idx_values), length(k_seq))
        vals[seq_len(idx_len)] <- as.numeric(all_idx_values[seq_len(idx_len)])
      }
      
    # 行列 or データフレームの場合（gap など一部の index がこれになることがある）
    } else if (is.matrix(all_idx_values) || is.data.frame(all_idx_values)) {
      # ひとまず「1 列目」をその index の値とみなす（厳密にやるなら列名で分岐）
      vec <- as.numeric(all_idx_values[, 1])
      idx_len <- min(length(vec), length(k_seq))
      vals[seq_len(idx_len)] <- vec[seq_len(idx_len)]
    }
  } else {
    vals <- rep(NA_real_, length(k_seq))
  }
  
  ## -----------------------------
  ## 3) row に詰めて返す
  ## -----------------------------
  row$best_k <- best_k
  row$value_best_k <- value_best_k
  row[, paste0("k", k_seq)] <- t(vals)
  
  return(row)
}


# Main processing for one dataset (A/B/C/D) & one k-max condition
process_dataset_one_kmax <- function(ds, ROOT, mds_type_for_clustering,
                                     mds_dim_for_clustering, nb_indices,
                                     min_nc, max_nc, k_tag, current_date) {
  name       <- ds$name
  prefix     <- ds$prefix
  input_dir  <- ds$input_dir
  output_dir <- ds$output_dir
  
  cat("\n----------------------------------------\n")
  cat("[INFO] Dataset:", name, " (prefix =", prefix, ")  k_max =", max_nc, "\n")
  cat("  Input dir :", input_dir, "\n")
  cat("  Output dir:", output_dir, "\n")
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  
  ## 1) Read MDS coordinates
  mds_file <- find_latest_mds_file(input_dir, prefix, mds_type_for_clustering)
  cat("[INFO] Using MDS file:", mds_file, "\n")
  mds_obj <- readRDS(mds_file)
  mds_coords <- get_mds_coords(mds_obj)
  
  # Keep only first mds_dim_for_clustering dimensions (or fewer if not available)
  n_dim_avail <- ncol(mds_coords)
  use_dim <- min(mds_dim_for_clustering, n_dim_avail)
  mds_mat <- mds_coords[, seq_len(use_dim), drop = FALSE]
  cat("[INFO] n_samples =", nrow(mds_mat),
      ", n_dims_used =", use_dim, "\n")
  
  if (is.null(rownames(mds_mat))) {
    sample_ids <- paste0("sample_", seq_len(nrow(mds_mat)))
    rownames(mds_mat) <- sample_ids
  } else {
    sample_ids <- rownames(mds_mat)
  }
  
  ## 2) Hierarchical clustering (ward.D2) — base structure
  dmat <- dist(mds_mat, method = "euclidean")
  hc <- hclust(dmat, method = "ward.D2")
  
  # Save hclust object（A〜D & 日付で 1 個だけあれば良いので、上書きでOK）
  hclust_outfile <- file.path(
    output_dir,
    sprintf("hclust_wardD2_%s_%s.rds", prefix, current_date)
  )
  saveRDS(hc, hclust_outfile)
  cat("[SAVED] hclust object:", hclust_outfile, "\n")
  
  ## 3) Cluster labels for k = min_nc..max_nc (from hclust)
  k_seq <- min_nc:max_nc
  labels_wide <- data.frame(
    sample_id = sample_ids,
    stringsAsFactors = FALSE
  )
  for (k in k_seq) {
    cl <- cutree(hc, k = k)
    labels_wide[[paste0("cluster_k", k)]] <- as.integer(cl[sample_ids])
  }
  
  labels_wide_file <- file.path(
    output_dir,
    sprintf("nbclust_labels_%s_%s_%s_wide.csv", prefix, k_tag, current_date)
  )
  write.csv(labels_wide, labels_wide_file, row.names = FALSE)
  cat("[SAVED] Cluster labels (wide):", labels_wide_file, "\n")
  
  # Long format (sample_id, k, cluster_label)
  labels_long <- do.call(
    rbind,
    lapply(k_seq, function(k) {
      data.frame(
        sample_id = sample_ids,
        k = k,
        cluster_label = labels_wide[[paste0("cluster_k", k)]],
        stringsAsFactors = FALSE
      )
    })
  )
  labels_long_file <- file.path(
    output_dir,
    sprintf("nbclust_labels_%s_%s_%s_long.csv", prefix, k_tag, current_date)
  )
  write.csv(labels_long, labels_long_file, row.names = FALSE)
  cat("[SAVED] Cluster labels (long):", labels_long_file, "\n")
  
  ## 4) NbClust: run for each index separately and build summary
  cat("[INFO] Running NbClust for indices (k from", min_nc, "to", max_nc, "):",
      paste(nb_indices, collapse = ", "), "\n")
  
  summary_rows <- list()
  for (idx in nb_indices) {
    cat("  - index:", idx, " ... ")
    nb_res <- run_nbclust_for_index(
      data_mat = mds_mat,
      index_name = idx,
      min_nc = min_nc,
      max_nc = max_nc
    )
    # Build summary row
    row <- build_index_summary_row(nb_res, idx, min_nc, max_nc)
    summary_rows[[idx]] <- row
    cat("done (best_k =", row$best_k, ")\n")
  }
  
  summary_df <- do.call(rbind, summary_rows)
  
  # wide: 1行1 index, k2..kmax の値を含む
  nb_summary_file <- file.path(
    output_dir,
    sprintf("nbclust_summary_%s_%s_%s.csv", prefix, k_tag, current_date)
  )
  write.csv(summary_df, nb_summary_file, row.names = FALSE)
  cat("[SAVED] NbClust index summary (wide):", nb_summary_file, "\n")
  
  # long: index, k, value （★ 各 k 時点の指標値をそのまま出力）
  k_cols <- paste0("k", min_nc:max_nc)
  summary_long <- do.call(
    rbind,
    lapply(seq_len(nrow(summary_df)), function(i) {
      idx_name <- summary_df$index[i]
      vals <- as.numeric(summary_df[i, k_cols])
      data.frame(
        index = idx_name,
        k     = min_nc:max_nc,
        value = vals,
        stringsAsFactors = FALSE
      )
    })
  )
  nb_summary_long_file <- file.path(
    output_dir,
    sprintf("nbclust_indices_long_%s_%s_%s.csv", prefix, k_tag, current_date)
  )
  write.csv(summary_long, nb_summary_long_file, row.names = FALSE)
  cat("[SAVED] NbClust index values (long):", nb_summary_long_file, "\n")
  
  cat("[INFO] Finished dataset:", name, " /", k_tag, "\n")
}


## -----------------------------
## 2. Dataset definitions (A–D)
## -----------------------------

datasets <- list(
  A = list(
    name       = "A_OH_plus_others",
    prefix     = "A",
    input_dir  = file.path(ROOT, "data/calc_core/01_distance_mds/A_OH_plus_others"),
    output_dir = file.path(ROOT, "data/calc_core/02_clustering/A_OH_plus_others")
  ),
  B = list(
    name       = "B_FP_plus_others",
    prefix     = "B",
    input_dir  = file.path(ROOT, "data/calc_core/01_distance_mds/B_FP_plus_others"),
    output_dir = file.path(ROOT, "data/calc_core/02_clustering/B_FP_plus_others")
  ),
  C = list(
    name       = "C_OH_only",
    prefix     = "C",
    input_dir  = file.path(ROOT, "data/calc_core/01_distance_mds/C_OH_only"),
    output_dir = file.path(ROOT, "data/calc_core/02_clustering/C_OH_only")
  ),
  D = list(
    name       = "D_FP_only",
    prefix     = "D",
    input_dir  = file.path(ROOT, "data/calc_core/01_distance_mds/D_FP_only"),
    output_dir = file.path(ROOT, "data/calc_core/02_clustering/D_FP_only")
  )
)


## -----------------------------
## 3. Main loop
## -----------------------------

cat("====================================================\n")
cat(" STEP2: Clustering on precomputed MDS coordinates\n")
cat(" ROOT =", ROOT, "\n")
cat(" MDS type for clustering :", mds_type_for_clustering, "\n")
cat(" MDS dims used           :", mds_dim_for_clustering, "\n")
cat(" NbClust indices         :", paste(nb_indices, collapse = ", "), "\n")
cat(" k-max list              :", paste(k_max_list, collapse = ", "), "\n")
cat("====================================================\n")

for (ds_name in names(datasets)) {
  ds <- datasets[[ds_name]]
  for (k_max in k_max_list) {
    k_tag <- paste0("kmax", k_max)
    process_dataset_one_kmax(
      ds  = ds,
      ROOT = ROOT,
      mds_type_for_clustering = mds_type_for_clustering,
      mds_dim_for_clustering  = mds_dim_for_clustering,
      nb_indices = nb_indices,
      min_nc = min_nc,
      max_nc = k_max,
      k_tag = k_tag,
      current_date = current_date
    )
  }
}

cat("\n[INFO] All datasets & k-max conditions finished.\n")


 STEP2: Clustering on precomputed MDS coordinates
 ROOT = /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127 
 MDS type for clustering : isoMDS 
 MDS dims used           : 10 
 NbClust indices         : silhouette, dunn, gap, ch, db, ptbiserial 
 k-max list              : 10, 25, 50 

----------------------------------------
[INFO] Dataset: A_OH_plus_others  (prefix = A )  k_max = 10 
  Input dir : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others 
  Output dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others 


[INFO] Candidate MDS files in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others :



[1] "mds_isoMDS_A_20251128.rds"


[INFO] Selected latest MDS file: mds_isoMDS_A_20251128.rds



[INFO] Using MDS file: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others/mds_isoMDS_A_20251128.rds 
[INFO] n_samples = 1046 , n_dims_used = 2 
[SAVED] hclust object: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/hclust_wardD2_A_20251128.rds 
[SAVED] Cluster labels (wide): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/nbclust_labels_A_kmax10_20251128_wide.csv 
[SAVED] Cluster labels (long): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/nbclust_labels_A_kmax10_20251128_long.csv 
[INFO] Runni

...

[INFO] Selected latest MDS file: mds_isoMDS_A_20251128.rds



[INFO] Using MDS file: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others/mds_isoMDS_A_20251128.rds 
[INFO] n_samples = 1046 , n_dims_used = 2 
[SAVED] hclust object: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/hclust_wardD2_A_20251128.rds 
[SAVED] Cluster labels (wide): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/nbclust_labels_A_kmax25_20251128_wide.csv 
[SAVED] Cluster labels (long): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/02_clustering/A_OH_plus_others/nbclust_labels_A_kmax25_20251128_long.csv 
[INFO] Runni

[INFO] Candidate MDS files in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core/01_distance_mds/A_OH_plus_others :



[1] "mds_isoMDS_A_20251128.rds"


[INFO] Selected latest MDS file: mds_isoMDS_A_20251128.rds

